# Project 1 

## Author: Yefrid Cordoba

## ST - 554 Big data analysis

## Part I

In this part, a class`SparkDataCheck` was created to perform a data check for the airquality data from the UC Irvine ML repo, including different ways to create instances for the class, checking that the some numeric variable falls in a user defined range, or showing frequencies for 1 or 2 non-numeric variables.

First we initialize the Spark session

In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Then, the module that contains the class is imported into the session

In [20]:
import Project2_class_YC
import pandas as pd

In [21]:
import importlib
importlib.reload(Project2_class_YC)

<module 'Project2_class_YC' from '/home/jupyter-yacordob@ncsu.edu/Project2_class_YC.py'>

Now, an instance for the class is created from the csv file, which is saved ina folder called `Data`, relative to my working directory.

In [22]:
air_data = Project2_class_YC.SparkDataCheck.createfrom_csv(spark, 'Data/air.csv')

It is necessary to rename the columns that have notation that can be undertand in a different way by the program (e.g., `'PT08.S1(CO)'` to `'S1(CO)'` )

In [23]:
air_data.df = air_data.df.withColumnsRenamed({\
                                              'PT08.S1(CO)':'S1(CO)', 'PT08.S2(NMHC)':'S2(NMHC)', 'PT08.S3(NOx)':'S3(NOx)', \
                                              'PT08.S4(NO2)': 'S4(NO2)', 'PT08.S5(O3)':'S5(O3)'})

In [24]:
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-23 22:00:00|   1.6|  1272|      51|     6.5|  

26/03/23 13:12:13 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


Showinf the first 10 rows of the instance of the class created, from which the last row (index 9), contains missing values, which are shown as `-200` in the dataframe./
For that reason, it is necessary to replace those values in the dataframe using the method `df.replace()` to create the Null values recognizable by pySpark.

In [25]:
air_data.df = air_data.df.replace(-200, None)

Checking that the `-200` changed for `Null` (or None)in the dataframe.

In [26]:
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-23 22:00:00|   1.6|  1272|      51|     6.5|  

26/03/23 13:12:13 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


### Examples of the class methods

#### When all bounds are included for a numeric variable

First we are going to check when it is numeric and both of the limits are included in the method ()which should work whitout exeptions

In [27]:
air_data.create_range_boolean('NO2(GT)', lower = 90, upper = 115)

DataFrame[_c0: int, Date: string, Time: timestamp, CO(GT): double, S1(CO): int, NMHC(GT): int, C6H6(GT): double, S2(NMHC): int, NOx(GT): int, S3(NOx): int, NO2(GT): int, S4(NO2): int, S5(O3): int, T: double, RH: double, AH: double, Is_in_range: boolean]

In [28]:
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|Is_in_range|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|       true|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|       true|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|       true|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.786

26/03/23 13:12:13 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


It is clear that based on the range provided it included what it was supposed to, also the Null values were included as Null in the new column created.

#### When lower limit is missing

In this part we are going to check the code when lower bound is missing.

In [29]:
air_data.create_range_boolean('NO2(GT)', upper = 115)
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|Is_in_range|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|       true|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|       true|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|       true|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.786

26/03/23 13:12:13 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


Of course the values that were below the lower limit previously now are included in the range.

#### When upper limit is missing

Now the upper limit is not going to be included in the method.

In [30]:
air_data.create_range_boolean('NO2(GT)', lower = 90)
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|Is_in_range|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|       true|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|       true|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|       true|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.786

26/03/23 13:12:13 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


And all the values that are above the lower limit are included now.

#### Non-numeric column is added

The column date now is going to be the input for our method to test the printed message

In [31]:
air_data.create_range_boolean('Date', lower = 90)


The column Date is not a numeric column


DataFrame[_c0: int, Date: string, Time: timestamp, CO(GT): double, S1(CO): int, NMHC(GT): int, C6H6(GT): double, S2(NMHC): int, NOx(GT): int, S3(NOx): int, NO2(GT): int, S4(NO2): int, S5(O3): int, T: double, RH: double, AH: double, Is_in_range: boolean]

We see that there is no modification to the dataframe and the message is raised because the column `Date` is not numric type.

#### Checking summarization methods

First using both arguments for the summarization method.

In [32]:
air_data.get_min_max(column = 'CO(GT)', group = 'Date')

,Date,max_CO(GT),min_CO(GT)
0,9/2/2004,5.2,0.6
1,12/26/2004,3.8,0.3
2,2/18/2005,4.8,0.4
3,10/10/2004,3.7,0.7
4,10/11/2004,4.8,0.4
...,...,...,...
386,1/23/2005,0.8,0.8
387,6/28/2004,5.6,0.4
388,8/16/2004,2.1,0.3
389,12/20/2004,4.1,0.6


When just one of the arguments is given

In [33]:
air_data.get_min_max(column = 'S2(NMHC)')

,max_S2(NMHC),min_S2(NMHC)
0,2214,383


Now trying to rise an error

In [34]:
air_data.get_min_max(column = 'Date')

The column Date is not a numeric column


It is seen that the code works how is intended to.

### Reading as a `pd.DataFrame`

We are going to imort the data now as a pandas dataframe, and from that dataframe we are going to create and instance of the class using the `@classmethod` created in the class

In [35]:
pd_air_data = pd.read_csv('Data/air.csv')

In [36]:
f_air_data = Project2_class_YC.SparkDataCheck.createfrom_pandas(spark, pd_air_data)

#### Example for the methods

We are going to check the count for the variable `Date` in the dataframe

In [39]:
f_air_data.get_counts('Date')

,Date,Count
0,3/11/2004,24
1,3/12/2004,24
2,3/10/2004,6
3,3/13/2004,24
4,3/15/2004,24
...,...,...
386,3/30/2005,24
387,3/31/2005,24
388,4/3/2005,24
389,4/4/2005,15


This table shows what is the number of measurements per day during the time the digital nose was recording air quality data from the Italian city.